# BRKGA Foco — Flow Shop Scheduling (NPFS)

Comparação das 3 variantes do BRKGA (fixo_default, fixo_optuna, fuzzy)
em **todas as instâncias 16×** (16 jobs, 2–5 máquinas, grupos 2–4).

**Fluxo:**
1. Tuning com Optuna na instância alvo 16×5
2. Calibração das referências fuzzy
3. Loop comparativo: 60 instâncias × 5 seeds × 4 variantes
4. Resultado consolidado em CSV (checkpoint incremental)
5. Tabela com makespan, tempo, lower bound e gap

## 0. Setup — Instalação e Imports

In [ ]:
# Instala dependencias (roda uma vez)
!pip install -q pymoo scikit-fuzzy optuna numpy pandas
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Dependencias instaladas.')

In [ ]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Defina aqui o caminho da sua pasta no Drive
# Exemplo: ROOT = '/content/drive/MyDrive/FlowShopScheduling'
ROOT = '/content/drive/MyDrive/FlowShopScheduling'

import sys, os, time, json, csv
sys.path.insert(0, ROOT)

from pathlib import Path
from typing import List, Dict, Optional, Tuple, Callable

import numpy as np
import pandas as pd
import optuna

DATA_DIR = Path(ROOT) / 'data' / 'Small'
RESULTADOS_DIR = Path(ROOT) / 'resultados_brkga_foco'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Resultados em: {RESULTADOS_DIR}')

In [ ]:
# Imports do projeto
from src.flowshop_engine import FlowShopEngine
from src.solvers.adapter import rodar_brkga, rodar_brkga_scs, SolverResult
from src.solvers.brkga_solver import (
    rodar_optuna,
    analisar_melhor_configuracao,
    construir_referencia_a_partir_da_trajetoria,
)

print('Imports concluídos.')

## 1. Catálogo de Instâncias — 60 instâncias 16×

Extrai todas as instâncias com 16 jobs do diretório `data/Small/`.
Parseia o nome do arquivo: `I_{grupo}_{n_jobs}_{n_machines}_{id}.txt`.

In [ ]:
def parse_nome_instancia(nome: str) -> Dict:
    """
    Converte 'I_2_16_5_1.txt' -> {grupo: 2, n_jobs: 16, n_machines: 5, id: 1}
    """
    stem = nome.replace('.txt', '')
    partes = stem.split('_')
    return {
        'grupo': int(partes[1]),
        'n_jobs': int(partes[2]),
        'n_machines': int(partes[3]),
        'id': int(partes[4]),
        'arquivo': nome,
    }


def construir_catalogo_16x(data_dir: Path) -> pd.DataFrame:
    """
    Retorna DataFrame com todas as instancias de 16 jobs.
    """
    todos = sorted(data_dir.glob('I_*_16_*.txt'))
    registros = []
    for f in todos:
        info = parse_nome_instancia(f.name)
        info['path'] = str(f)
        registros.append(info)

    df = pd.DataFrame(registros)
    df = df.sort_values(['grupo', 'n_machines', 'id']).reset_index(drop=True)
    df['rotulo'] = df.apply(
        lambda r: f'{r["n_jobs"]}x{r["n_machines"]}_g{r["grupo"]}_id{r["id"]}', axis=1
    )
    return df


catalogo = construir_catalogo_16x(DATA_DIR)
print(f'Total de instancias 16x: {len(catalogo)}')
print()
print('Distribuicao:')
for m in sorted(catalogo['n_machines'].unique()):
    qtd = len(catalogo[catalogo['n_machines'] == m])
    print(f'  16x{m}: {qtd} instancias')
print()
print('10 primeiras:')
print(catalogo.head(10).to_string(index=False))

## 2. Configuração Global

In [ ]:
# Parametros do BRKGA (padrao fabrica)
POP_SIZE = 100
N_ELITES_DEFAULT = 20
N_OFFSPRINGS_DEFAULT = 70
N_MUTANTS_DEFAULT = 10
BIAS_DEFAULT = 0.7

# Geracoes para o experimento comparativo
N_GEN = 100  # 100 gen x 100 pop = 10.000 avaliacoes

# Sementes para o experimento
N_SEEDS = 5
SEED_INICIO = 100

# Objetivo
OBJECTIVE = 'Makespan'

# Tuning
INSTANCIA_ALVO = 'I_2_16_5_1.txt'   # alvo principal (v5)
N_TRIALS_OPTUNA = 15
SEEDS_TUNING = (1, 2, 3)            # 3 seeds por trial

print('Configuracao:')
print(f'  Populacao: {POP_SIZE} ({N_ELITES_DEFAULT} elites, {N_OFFSPRINGS_DEFAULT} offsprings, {N_MUTANTS_DEFAULT} mutantes)')
print(f'  Geracoes: {N_GEN}')
print(f'  Avaliacoes/exec: {N_GEN * POP_SIZE}')
print(f'  Seeds: {N_SEEDS}  ({SEED_INICIO}..{SEED_INICIO + N_SEEDS - 1})')
print(f'  Objetivo: {OBJECTIVE}')
print(f'  Instancia alvo tuning: {INSTANCIA_ALVO}')
print(f'  Trials Optuna: {N_TRIALS_OPTUNA}')

## 3. Tuning com Optuna — Instância Alvo 16×5

Roda o Optuna na instância alvo (`I_2_16_5_1.txt`)
para encontrar os melhores parâmetros: `frac_elites`, `frac_mutantes`, `bias`.
O resultado é salvo em JSON para reuso futuro.

In [ ]:
# Carrega a engine da instancia alvo
engine_alvo = FlowShopEngine(
    *FlowShopEngine.carregar_instancia_txt(
        str(DATA_DIR / INSTANCIA_ALVO)
    )
)
print(f'Engine alvo: {engine_alvo.n_jobs} jobs, {engine_alvo.n_machines} maquinas')

In [ ]:
print('>>> Iniciando tuning com Optuna...')
t0_tuning = time.time()

study, melhor_config = rodar_optuna(
    engine_alvo,
    pop_size=POP_SIZE,
    n_gen=N_GEN,           # 100 gen durante o tuning tambem
    seeds=SEEDS_TUNING,
    n_trials=N_TRIALS_OPTUNA,
    seed_optuna=0,
)

elapsed_tuning = time.time() - t0_tuning
print(f'Tuning concluido em {elapsed_tuning:.1f}s')
print(f'Melhor custo: {study.best_value:.2f}')
print(f'Melhor config: {melhor_config}')

# Salva config em JSON
config_path = RESULTADOS_DIR / 'config_optuna.json'
with open(config_path, 'w') as f:
    json.dump(melhor_config, f, indent=2)
print(f'Config salva em: {config_path}')

## 4. Calibração das Referências Fuzzy

Usa a melhor config do Optuna para gerar as trajetórias médias
de diversidade e qualidade. Essas trajetórias servem como
referência para os controladores fuzzy, seguindo a metodologia
original do aluno_4.

In [ ]:
print('>>> Calibrando referencias fuzzy...')
t0_ref = time.time()

_, traj_div, traj_qual, resumo_calib = analisar_melhor_configuracao(
    engine_alvo,
    melhor_config,
    n_gen=N_GEN,
    seeds=(10, 11, 12, 13, 14),  # 5 seeds para a calibracao
)

ref_diversidade = construir_referencia_a_partir_da_trajetoria(traj_div)
ref_qualidade = construir_referencia_a_partir_da_trajetoria(traj_qual)

elapsed_ref = time.time() - t0_ref
print(f'Calibracao concluida em {elapsed_ref:.1f}s')
print(f'Makespan medio na calibracao: {resumo_calib["makespan_medio"]:.1f}')
print(f'  Trajetoria diversidade (10 primeiros): {np.round(traj_div[:10], 4)}')
print(f'  Trajetoria qualidade (10 primeiros):   {np.round(traj_qual[:10], 4)}')

## 5. Fábrica de Variantes (closures)

Cada variante é uma callable `(engine, seed) -> SolverResult`.
O futuro `brkga-scs` implementa a mesma assinatura e entra aqui.

In [ ]:
def fabricar_variante_fixo_default() -> Callable:
    """BRKGA com parametros de fabrica (20/70/10, bias 0.7)."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fixo_default',
            n_gen=N_GEN,
            n_elites=N_ELITES_DEFAULT,
            n_offsprings=N_OFFSPRINGS_DEFAULT,
            n_mutants=N_MUTANTS_DEFAULT,
            bias=BIAS_DEFAULT,
        )
    return _run


def fabricar_variante_fixo_optuna(config: dict) -> Callable:
    """BRKGA com parametros encontrados pelo Optuna."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fixo_optuna',
            n_gen=N_GEN,
            n_elites=config['n_elites'],
            n_offsprings=config['n_offsprings'],
            n_mutants=config['n_mutants'],
            bias=config['bias'],
        )
    return _run


def fabricar_variante_fuzzy(
    config: dict,
    ref_div: Callable,
    ref_qual: Callable,
) -> Callable:
    """BRKGA adaptativo com controle fuzzy, partindo da config do Optuna."""
    def _run(engine, seed):
        return rodar_brkga(
            engine, seed=seed, objective=OBJECTIVE,
            version='fuzzy',
            n_gen=N_GEN,
            n_elites=config['n_elites'],
            n_offsprings=config['n_offsprings'],
            n_mutants=config['n_mutants'],
            bias=config['bias'],
            ref_diversidade=ref_div,
            ref_qualidade=ref_qual,
        )
    return _run


def fabricar_variante_brkga_scs() -> Callable:
    """BRKGA-SCS (Clustering Search) com parametros padrao do notebook original."""
    def _run(engine, seed):
        return rodar_brkga_scs(
            engine, seed=seed, n_gen=N_GEN,
            tam_pop=POP_SIZE, pe=0.2, pm=0.15, rho=0.7,
            raio_cluster=0.5, z_limiar=1.5, votos_limiar=5,
        )
    return _run


# Monta o dicionario de variantes
variantes = {
    'fixo_default': fabricar_variante_fixo_default(),
    'fixo_optuna': fabricar_variante_fixo_optuna(melhor_config),
    'fuzzy': fabricar_variante_fuzzy(melhor_config, ref_diversidade, ref_qualidade),
    'brkga_scs': fabricar_variante_brkga_scs(),
}

print(f'Variantes disponiveis: {list(variantes.keys())}')

## 6. Loop Comparativo — 60 instâncias × 5 seeds × 4 variantes

Para cada instância:
- Carrega a engine
- Roda cada variante × seed
- Salva resultados incrementalmente no CSV

O CSV contém: `instancia, grupo, n_jobs, n_machines, metodo, seed,
makespan, tardiness, cfo_count, tempo_s, lower_bound, gap`
mais os makespans intermediários a cada 20 gerações
(gen 20, 40, 60, 80, 100) para análise de convergência futura.

In [ ]:
def amostrar_historico(history: list, n_gen: int, passo: int = 20) -> dict:
    """Extrai o makespan nos marcos gen=passo, 2*passo, ..., n_gen."""
    amostras = {}
    for gen in range(passo, n_gen + 1, passo):
        idx = min(gen - 1, len(history) - 1)
        amostras[f'mks_gen{gen}'] = float(history[idx])
    return amostras


# Schema fixo do CSV
CAMPOS_CSV = [
    'instancia', 'grupo', 'n_jobs', 'n_machines',
    'metodo', 'seed',
    'makespan', 'tardiness', 'cfo_count', 'tempo_s',
    'lower_bound', 'gap',
]
# Campos de convergencia (a cada 20 geracoes)
CAMPOS_CONVERGENCIA = [f'mks_gen{g}' for g in range(20, N_GEN + 1, 20)]
CAMPOS_CSV.extend(CAMPOS_CONVERGENCIA)

CSV_PATH = RESULTADOS_DIR / 'resultados_brkga_foco.csv'
print(f'CSV de saida: {CSV_PATH}')
print(f'Campos: {CAMPOS_CSV}')

In [ ]:
print('=' * 72)
print('INICIANDO LOOP COMPARATIVO')
print(f'{len(catalogo)} instancias x {len(variantes)} variantes x {N_SEEDS} seeds')
print(f'Total: {len(catalogo) * len(variantes) * N_SEEDS} execucoes')
print('=' * 72)

t0_total = time.time()

# Flag para escrever header apenas na primeira leva
csv_header_escrito = False

for idx_inst, (_, row) in enumerate(catalogo.iterrows()):
    inst_label = row['rotulo']
    inst_path = row['path']
    n_jobs = row['n_jobs']
    n_mach = row['n_machines']
    grupo = row['grupo']

    # Carrega engine uma vez por instancia
    nj, nm, pt, dd = FlowShopEngine.carregar_instancia_txt(inst_path)
    engine = FlowShopEngine(nj, nm, pt, dd)

    for metodo, fn_variante in variantes.items():
        for seed in range(SEED_INICIO, SEED_INICIO + N_SEEDS):
            # LOG de progresso detalhado
            print(
                f'[{idx_inst + 1}/{len(catalogo)}] '
                f'Inst: {row["arquivo"]} ({n_jobs}j x {n_mach}m, g{grupo}) '
                f'| Var: {metodo} '
                f'| Seed: {seed} ...',
                end=' '
            )

            try:
                t0_exec = time.time()
                result = fn_variante(engine, seed)
                elapsed = time.time() - t0_exec

                # Monta linha do CSV
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float(result.best_cost),
                    'tardiness': float(result.tardiness),
                    'cfo_count': int(result.engine_cfo_count),
                    'tempo_s': round(elapsed, 3),
                    'lower_bound': float(result.lower_bound),
                    'gap': round(result.gap_percent(), 2),
                }
                # Amostras de convergencia
                amostras = amostrar_historico(result.history, N_GEN, 20)
                linha.update(amostras)

                print(f'OK  Makespan: {result.best_cost:.1f} ({elapsed:.2f}s)')

            except Exception as e:
                print(f'ERRO: {e}')
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float('nan'),
                    'tardiness': float('nan'),
                    'cfo_count': 0,
                    'tempo_s': float('nan'),
                    'lower_bound': float('nan'),
                    'gap': float('nan'),
                }
                for c in CAMPOS_CONVERGENCIA:
                    linha[c] = float('nan')

            # Escreve no CSV incremental (abre/fecha a cada linha = seguro)
            escrever_header = not csv_header_escrito
            with open(CSV_PATH, 'w' if escrever_header else 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=CAMPOS_CSV)
                if escrever_header:
                    writer.writeheader()
                    csv_header_escrito = True
                writer.writerow(linha)

elapsed_total = time.time() - t0_total
print('=' * 72)
print(f'LOOP CONCLUIDO em {elapsed_total:.1f}s ({elapsed_total/60:.1f}min)')
print(f'Resultados salvos em: {CSV_PATH}')

## 7. Resumo Final

Carrega o CSV consolidado e exibe a tabela comparativa
agregada por instância × método.

Colunas: `makespan` (média, desvio, melhor, pior),
`tempo_medio`, `lower_bound`, `gap_medio`.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total de linhas no CSV: {len(df)}')
print()

print('=' * 120)
print('TABELA COMPARATIVA — Makespan medio por Instancia x Metodo')
print('=' * 120)

# Tabela detalhada por instancia + metodo
agg = df.groupby(['instancia', 'n_jobs', 'n_machines', 'metodo']).agg(
    makespan_medio=('makespan', 'mean'),
    desvio=('makespan', 'std'),
    melhor=('makespan', 'min'),
    pior=('makespan', 'max'),
    tempo_medio=('tempo_s', 'mean'),
    gap_medio=('gap', 'mean'),
    lower_bound=('lower_bound', 'first'),
).reset_index()
agg['makespan_medio'] = agg['makespan_medio'].round(1)
agg['desvio'] = agg['desvio'].round(1)
agg['tempo_medio'] = agg['tempo_medio'].round(3)
agg['gap_medio'] = agg['gap_medio'].round(2)

print(agg.to_string(index=False))
print()

# Resumo global (agregando todas as instancias)
global_agg = df.groupby('metodo').agg(
    makespan_medio=('makespan', 'mean'),
    desvio_std=('makespan', 'std'),
    melhor_global=('makespan', 'min'),
    tempo_medio=('tempo_s', 'mean'),
    gap_medio=('gap', 'mean'),
).reset_index()
global_agg['makespan_medio'] = global_agg['makespan_medio'].round(1)
global_agg['desvio_std'] = global_agg['desvio_std'].round(1)
global_agg['tempo_medio'] = global_agg['tempo_medio'].round(3)
global_agg['gap_medio'] = global_agg['gap_medio'].round(2)

print('=' * 80)
print('RESUMO GLOBAL (todas as 60 instancias, todas as seeds)')
print('=' * 80)
print(global_agg.to_string(index=False))
print()
print(f'Arquivo salvo em: {CSV_PATH}')

In [ ]:
print('Notebook concluido com sucesso.')
print(f'Resultados em: {RESULTADOS_DIR}/')